In [58]:
import pandas as pd
import numpy as np
import yfinance as yf
import statsmodels.api as sm
import re
import nltk
from nltk.corpus import stopwords

# Κατεβάζουμε το λεξικό με τα stopwords (τρέχει μια φορά)
nltk.download('stopwords', quiet=True)


True

In [59]:
# Φόρτωση του αρχείου
df = pd.read_csv(r"..\data\CBS_dataset_v1.0.csv")



In [60]:
# Εμφάνιση των πρώτων γραμμών
df.head()

,URL,PDF,Title,Subtitle,Date,Authorname,Role,Gender,CentralBank,Country,text,text_original,Filename,Language,Source
0,https://www.cbaruba.org/readBlob.do?id=10756,NaN,President speech Managing the Economy as if th...,NaN,2021-12-08,Jeanette R Semeleer,Governor,Female,Central Bank of Aruba,ABW,Managing the Economy as if the Future Really M...,NaN,abw_10756,English,CB websites
1,https://www.cbaruba.org/readBlob.do?id=7515,NaN,Speech President of the CBA 4th Annual Future ...,NaN,2019-11-01,Jeanette R Semeleer,Governor,Female,Central Bank of Aruba,ABW,Safeguarding our Future: Strategies for an Aru...,NaN,abw_7515,English,CB websites
2,https://www.cbaruba.org/readBlob.do?id=7518,NaN,Speech Symposium President Semeleer CBA,NaN,2019-09-06,Jeanette R Semeleer,Governor,Female,Central Bank of Aruba,ABW,FOSTERING ECONOMIC RESILIENCE IN ARUBA; FROM R...,NaN,abw_7518,English,CB websites
3,https://www.cbaruba.org/readBlob.do?id=7548,NaN,Integrity Koninkrijk Seminar,NaN,2016-10-28,Jeanette R Semeleer,Governor,Female,Central Bank of Aruba,ABW,"Presentation by Mrs Jeanette R. Semeleer, Pres...","Voordracht door mevrouw Jeanette R. Semeleer, ...",abw_7548,Dutch,CB websites
4,https://www.cbaruba.org/readBlob.do?id=7554,NaN,Speech by the President at the BES seminar hel...,NaN,2010-03-29,Jeanette R Semeleer,Governor,Female,Central Bank of Aruba,ABW,Ongoing changes in the supervisory landscape a...,NaN,abw_7554,English,CB websites


In [61]:
# Κρατάμε μόνο τις 4 στήλες pou einai xrhsimes
df = df[['Date', 'text', 'Country', 'CentralBank']]

# elenxoyme an yparxoyn kena text
df = df.dropna(subset=['text'])

# taxinomo apo thn mikroterh mexri thn megaluterh hmeromhnia
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')




In [62]:
# Ορίζουμε το σετ των stopwords
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if not isinstance(text, str): return ""
    text = text.lower() # Όλα μικρά
    text = re.sub(r'\d+', '', text) # Αφαίρεση αριθμών
    text = re.sub(r'[^\w\s]', '', text) # Αφαίρεση σημείων στίξης
    
    # Χωρίζουμε το κείμενο σε λέξεις και κρατάμε ΜΟΝΟ όσες δεν είναι stopwords
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    
    # Τα ξαναενώνουμε σε μια καθαρή πρόταση
    return ' '.join(filtered_words)

In [ ]:
#Λεξικό με λέξεις ρίσκου/αβεβαιότητας
uncertainty_words = ['uncertainty', 'uncertain', 'risk', 'risks', 'volatile', 
                     'volatility', 'might', 'possibly', 'unclear', 'headwinds', 'threat']
pattern = re.compile(r'\b(?:' + '|'.join(uncertainty_words) + r')\b')
#Για κάθε ομιλία, υπολογίζει έναν δείκτη πυκνότητας analoga me to poses lexeis tou keimenoy tairiazoyn me tis lexeis tou lexikoy
def calc_uncertainty(clean_text_input):
    words = len(clean_text_input.split())
    if words == 0: return 0
    matches = len(pattern.findall(clean_text_input))
    return (matches / words) * 1000  # Εμφανίσεις ανά 1000 ΟΥΣΙΑΣΤΙΚΕΣ λέξεις




Καθάρισμα κειμένων (με αφαίρεση Stopwords)... Αυτό ίσως πάρει 1-2 λεπτά!
Υπολογισμός Δείκτη Αβεβαιότητας...
Ο αναβαθμισμένος Δείκτης Αβεβαιότητας υπολογίστηκε επιτυχώς ανά μήνα!


In [ ]:
print("Καθάρισμα κειμένων (με αφαίρεση Stopwords)... Αυτό ίσως πάρει 1-2 λεπτά!")
df['clean_text'] = df['text'].apply(clean_text)

print("Υπολογισμός Δείκτη Αβεβαιότητας...")
df['Uncertainty_Index'] = df['clean_text'].apply(calc_uncertainty)

# Ομαδοποίηση ανά μήνα (εύρεση μέσου όρου)
df.set_index('Date', inplace=True)
monthly_speeches = df['Uncertainty_Index'].resample('ME').mean().dropna()

print("Ο αναβαθμισμένος Δείκτης Αβεβαιότητας υπολογίστηκε επιτυχώς ανά μήνα!")

In [ ]:
print("Λήψη χρηματοοικονομικών δεδομένων (VIX) από το Yahoo Finance...")
start_date = monthly_speeches.index.min().strftime('%Y-%m-%d')
end_date = monthly_speeches.index.max().strftime('%Y-%m-%d')

# Λήψη δεδομένων VIX
vix_data = yf.download('^VIX', start=start_date, end=end_date, progress=False)
# Κρατάμε το κλείσιμο του μήνα
monthly_vix = vix_data['Close'].resample('ME').mean()

# Ένωση των δύο μεταβλητών σε ένα τελικό DataFrame
final_data = pd.concat([monthly_speeches, monthly_vix], axis=1).dropna()

# Διορθώνουμε τα ονόματα των στηλών
if isinstance(final_data.columns, pd.MultiIndex):
    final_data.columns = ['Uncertainty_Index', 'VIX_Close']
else:
    final_data.columns = ['Uncertainty_Index', 'VIX_Close']

print("Η ένωση ολοκληρώθηκε! Έτοιμοι για τη στατιστική ανάλυση.")
final_data.head()

Λήψη χρηματοοικονομικών δεδομένων (VIX) από το Yahoo Finance...
Price           Close       High        Low       Open Volume
Ticker           ^VIX       ^VIX       ^VIX       ^VIX   ^VIX
Date                                                         
1990-01-02  17.240000  17.240000  17.240000  17.240000      0
1990-01-03  18.190001  18.190001  18.190001  18.190001      0
1990-01-04  19.219999  19.219999  19.219999  19.219999      0
1990-01-05  20.110001  20.110001  20.110001  20.110001      0
1990-01-08  20.260000  20.260000  20.260000  20.260000      0
Η ένωση ολοκληρώθηκε! Έτοιμοι για τη στατιστική ανάλυση.


,Uncertainty_Index,VIX_Close
Date,,
1990-01-31,2.489195,23.347273
1990-02-28,5.895222,23.262632
1990-03-31,6.179551,20.062273
1990-04-30,4.490863,21.403500
1990-05-31,8.748788,18.097727


In [66]:
# 1. Υπολογισμός Συσχέτισης (Pearson Correlation)
correlation = final_data['Uncertainty_Index'].corr(final_data['VIX_Close'])
print(f"--- ΣΥΣΧΕΤΙΣΗ (Correlation) ---")
print(f"Συντελεστής Συσχέτισης Pearson: {correlation:.4f}\n")

# 2. Υπολογισμός OLS Regression
X = sm.add_constant(final_data['Uncertainty_Index']) # Ανεξάρτητη: Δείκτης Αβεβαιότητας
Y = final_data['VIX_Close']                          # Εξαρτημένη: VIX

# Τρέχουμε το μοντέλο OLS
model = sm.OLS(Y, X).fit()

print(f"--- ΑΠΟΤΕΛΕΣΜΑΤΑ OLS REGRESSION ---")
print(model.summary())

--- ΣΥΣΧΕΤΙΣΗ (Correlation) ---
Συντελεστής Συσχέτισης Pearson: 0.0899

--- ΑΠΟΤΕΛΕΣΜΑΤΑ OLS REGRESSION ---
                            OLS Regression Results                            
Dep. Variable:              VIX_Close   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                  0.006
Method:                 Least Squares   F-statistic:                     3.307
Date:                Tue, 17 Mar 2026   Prob (F-statistic):             0.0697
Time:                        20:02:57   Log-Likelihood:                -1401.6
No. Observations:                 408   AIC:                             2807.
Df Residuals:                     406   BIC:                             2815.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
----------------